# October Task #6/November Task #2 — Conversation-quality model experiments

Try to improve the conversation-level derailment model on CGA-CMV. This notebook is an experiment template, not a final implementation: each proposed variation should be reviewed by the team before being adopted.

The goal is to test whether additional information available before the outcome can improve the headline baseline while preserving the repository's evaluation rules.

The primary comparison is against **structural features + reply-score summaries**. The final test set must remain untouched until a model and configuration have been selected from development results.


## Issue context and fixed rules

The model predicts whether a conversation will remain civil or derail. The baseline setup remains fixed:

- hide the final comment, which is the outcome-bearing comment;
- use the dataset's train/validation/test split, keeping matched pairs together;
- exclude leaky deletion features;
- treat vote-based features as diagnostic only;
- compute reply-model scores without allowing a conversation's final reply, or its matched partner's final reply, into the fitting model;
- compare validation to validation during development and test to test only at the end.

The issue asks for at least one variation, confidence intervals, error analysis, a log of unsuccessful experiments, and a recommendation about whether the result belongs in the report as a conversation-health signal.


In [1]:
from pathlib import Path
import sys
import warnings

current = Path.cwd()
repo_root = current if (current / "artifacts").exists() else current.parent
sys.path.insert(0, str(repo_root / "artifacts"))
warnings.filterwarnings("ignore", category=RuntimeWarning)

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.pipeline import FeatureUnion, Pipeline

from conversation_quality import (
    CIVIL, DERAILED, UNCIVIL, REPLY_SCORE_COLUMNS, TEXT_FEATURE_COLUMNS,
    load_examples, load_replies, add_reply_scores, structural_columns, paired_accuracy,
    majority_baseline, feature_model, text_model,
)
from evaluation import evaluate_classifier
from standardize_convokit import iter_standardized
from structural_features import conversation_features

path = repo_root / "data" / "local" / "standardized-v1" / "conversations-gone-awry-cmv-corpus.jsonl"


def score(model, X, frame):
    """Label metrics via evaluation.py, plus ranking metrics for the quality score."""
    result = evaluate_classifier(model, X, frame["label"].tolist())
    quality = model.quality_scores(X)
    return result, quality, {
        "accuracy": result.accuracy,
        "macro_f1": result.macro_f1,
        "roc_auc": roc_auc_score(frame["label"] == CIVIL, quality),
        "paired_acc": paired_accuracy(quality, frame["label"].tolist(),
                                      frame["pair_id"].tolist(), frame["conversation_id"].tolist()),
    }

DATA_PATH = path
assert path.is_file(), f"Place the standardized CGA-CMV file at {path}"
class PipelineQualityModel:
    """Adapter giving a plain sklearn pipeline the repository model interface."""
    def __init__(self, pipeline):
        self.pipeline = pipeline

    def fit(self, X, labels):
        self.pipeline.fit(X, list(labels))
        return self

    def predict(self, X):
        return self.pipeline.predict(X)

    def quality_scores(self, X):
        civil_index = list(self.pipeline.classes_).index(CIVIL)
        return self.pipeline.predict_proba(X)[:, civil_index].tolist()

SEED = 42


## Data loading and the headline baseline

The next cell reconstructs the repository baseline. The blank cell after it is intentionally reserved for interpreting the observed validation results.


In [2]:
finals, openings = load_replies(path)
examples = add_reply_scores(load_examples(path), finals, openings)
train = examples.loc[examples["split"] == "train"].copy()
val = examples.loc[examples["split"] == "val"].copy()
test = examples.loc[examples["split"] == "test"].copy()

structural_cols = structural_columns(train, include_votes=False)
reply_cols = list(REPLY_SCORE_COLUMNS)
headline_cols = structural_cols + reply_cols

def civil_probability(model, X):
    return np.asarray(model.quality_scores(X))

headline = feature_model(headline_cols).fit(train, train["label"])
baseline_val = score(headline, val, val)[2]
pd.Series(baseline_val, name="headline baseline").round(4)


accuracy      0.6279
macro_f1      0.6279
roc_auc       0.6746
paired_acc    0.6959
Name: headline baseline, dtype: float64

## Conclusion

The headline baseline achieves 0.6279 accuracy and macro F1, ROC AUC 0.6746, and paired accuracy 0.6959 on validation. These metrics establish the development reference for the proposed variations.


## Candidate 1 — richer reply-score summaries

The baseline keeps only the minimum, mean, and most recent estimated civility of the opening replies. The following variation adds simple trajectory and spread terms derived only from those already leakage-controlled summaries. It tests whether the model benefits from distinguishing a consistently poor opening from one that deteriorates near the end.


In [3]:
def add_reply_summary_features(frame):
    out = frame.copy()
    out["reply_last_minus_min"] = out["reply_last_civil"] - out["reply_min_civil"]
    out["reply_last_minus_mean"] = out["reply_last_civil"] - out["reply_mean_civil"]
    out["reply_mean_minus_min"] = out["reply_mean_civil"] - out["reply_min_civil"]
    out["reply_last_is_min"] = np.isclose(out["reply_last_civil"], out["reply_min_civil"], equal_nan=False).astype(float)
    return out

train_rich = add_reply_summary_features(train)
val_rich = add_reply_summary_features(val)
rich_reply_cols = reply_cols + ["reply_last_minus_min", "reply_last_minus_mean", "reply_mean_minus_min", "reply_last_is_min"]
rich_model = feature_model(structural_cols + rich_reply_cols).fit(train_rich, train_rich["label"])
rich_val = score(rich_model, val_rich, val_rich)[2]
pd.DataFrame([baseline_val, rich_val], index=["headline", "rich reply summaries"]).round(4)


,accuracy,macro_f1,roc_auc,paired_acc
headline,0.6279,0.6279,0.6746,0.6959
rich reply summaries,0.6272,0.6271,0.6752,0.6930


## Conclusion

The richer reply-score summaries do not improve the baseline overall. Accuracy falls from 0.6279 to 0.6272, macro F1 from 0.6279 to 0.6271, and paired accuracy from 0.6959 to 0.6930. ROC AUC increases only slightly, from 0.6746 to 0.6752. This variation should be recorded as unsuccessful on its own.


## Candidate 2 — stabilize skewed structural features

Several structural variables are heavily right-skewed, especially elapsed time and reply gaps. The existing model standardizes them but does not change their scale. This variation adds `log1p` versions while retaining the original columns, allowing logistic regression to use both raw and compressed effects.


In [4]:
SKEWED_STRUCTURAL = ["n_utterances", "max_depth", "max_branching", "max_width", "duration_seconds", "median_reply_gap", "mean_reply_gap", "min_reply_gap", "max_reply_gap", "time_to_first_reply", "growth_rate_per_hour", "n_speakers", "self_reply_count"]

def add_log_structural_features(frame):
    out = frame.copy()
    for col in SKEWED_STRUCTURAL:
        if col in out:
            values = pd.to_numeric(out[col], errors="coerce").clip(lower=0)
            out[f"log1p_{col}"] = np.log1p(values)
    return out

train_log = add_log_structural_features(train_rich)
val_log = add_log_structural_features(val_rich)
log_cols = structural_cols + [f"log1p_{c}" for c in SKEWED_STRUCTURAL if c in train_log]
log_model = feature_model(log_cols + rich_reply_cols).fit(train_log, train_log["label"])
log_val = score(log_model, val_log, val_log)[2]
pd.DataFrame([baseline_val, rich_val, log_val], index=["headline", "rich reply summaries", "log structural + rich reply"]).round(4)


,accuracy,macro_f1,roc_auc,paired_acc
headline,0.6279,0.6279,0.6746,0.6959
rich reply summaries,0.6272,0.6271,0.6752,0.6930
log structural + rich reply,0.6345,0.6344,0.6872,0.7164


## Conclusion

Adding `log1p` versions of the skewed structural features produces the strongest validation result tested so far. Compared with the headline baseline, accuracy increases from 0.6279 to 0.6345, macro F1 from 0.6279 to 0.6344, ROC AUC from 0.6746 to 0.6872, and paired accuracy from 0.6959 to 0.7164. Because all four metrics improve, this is the provisional leading candidate for final evaluation.


## Candidate 3 — stronger opening-text model

The existing opening-text comparison uses word-level TF-IDF with word unigrams and bigrams. Character n-grams can capture spelling, punctuation, contractions, and hostile phrasing without requiring a large transformer run. This is a separate text experiment and should be compared against the numeric baseline before blending.


In [5]:
def word_char_text_model(class_weight=None):
    features = FeatureUnion([
        ("word", TfidfVectorizer(lowercase=True, ngram_range=(1, 2), min_df=2, sublinear_tf=True, max_features=150_000)),
        ("char", TfidfVectorizer(analyzer="char", lowercase=True, ngram_range=(3, 5), min_df=3, sublinear_tf=True, max_features=100_000)),
    ])
    return PipelineQualityModel(Pipeline([
        ("tfidf", features),
        ("classifier", LogisticRegression(max_iter=1500, class_weight=class_weight)),
    ]))

text_candidate = word_char_text_model()
text_candidate.fit(train["context_text"], train["label"])
text_result, text_quality, text_metrics = score(text_candidate, val["context_text"], val)
text_val = text_metrics
pd.DataFrame([text_val], index=["word + char TF-IDF opening text"]).round(4)


,accuracy,macro_f1,roc_auc,paired_acc
word + char TF-IDF opening text,0.6089,0.6089,0.6394,0.6594


## Conclusion

The word-and-character TF-IDF model underperforms the headline baseline on every reported validation metric: accuracy and macro F1 are 0.6089, ROC AUC is 0.6394, and paired accuracy is 0.6594. Opening text alone is therefore not competitive with the numeric baseline in this experiment.


## Candidate 4 — blend numeric and text scores

The headline model and opening-text model describe different signals. The blend below combines their estimated probability of `civil` without retraining on validation labels. The blend weight is selected using validation results; do not choose it using the test set.


In [6]:
headline_val_score = civil_probability(headline, val)
text_val_score = civil_probability(text_candidate, val["context_text"])
blend_rows = []
for headline_weight in np.linspace(0, 1, 11):
    blend_score = headline_weight * headline_val_score + (1 - headline_weight) * text_val_score
    pred = np.where(blend_score >= 0.5, CIVIL, DERAILED)
    blend_rows.append({"text_weight": 1 - headline_weight, "headline_weight": headline_weight, "accuracy": accuracy_score(val["label"], pred), "macro_f1": f1_score(val["label"], pred, average="macro"), "roc_auc": roc_auc_score(val["label"] == CIVIL, blend_score)})
pd.DataFrame(blend_rows).sort_values("macro_f1", ascending=False)


,text_weight,headline_weight,accuracy,macro_f1,roc_auc
8,0.2,0.8,0.629386,0.629370,0.677640
9,0.1,0.9,0.629386,0.629352,0.676426
10,0.0,1.0,0.627924,0.627867,0.674638
7,0.3,0.7,0.627193,0.627186,0.678025
5,0.5,0.5,0.626462,0.626446,0.676518
6,0.4,0.6,0.625731,0.625724,0.677813
4,0.6,0.4,0.621345,0.621305,0.672929
3,0.7,0.3,0.618421,0.618381,0.667530
2,0.8,0.2,0.616228,0.616154,0.659902
1,0.9,0.1,0.613304,0.613195,0.650450


## Conclusion

The best validation blend uses 80% headline-model score and 20% text-model score. It reaches accuracy 0.6294, macro F1 0.6294, and ROC AUC 0.6776. This is a small improvement over the headline baseline on validation, but it remains well below the log-transformed structural candidate's accuracy, macro F1, ROC AUC, and paired accuracy. The blend is not the leading candidate.


## Candidate 5 — forecasting cutoff

The main comparison uses every comment except the final one, as required by the baseline. This separate experiment asks how much signal is available earlier. It must be reported separately because changing the cutoff changes the prediction task.

This template uses structural features at a prefix cutoff. Reply-score summaries remain aligned with the current repository helper in the first pass; if this variation is promising, the team should add a cutoff-aware version of `add_reply_scores` so reply scores are also restricted to the prefix.


In [7]:
def prefix_structural_examples(path, first_n):
    rows = []
    for conversation in iter_standardized(path):
        source = conversation.metadata.get("source_metadata", {})
        real = [u for u in conversation.utterances if not u.metadata.get("is_synthetic")]
        ordered = sorted(real, key=lambda u: u.created_at) if all(u.created_at is not None for u in real) else real
        if not ordered:
            continue
        row = {"conversation_id": conversation.id, "pair_id": source.get("pair_id"), "split": source.get("split"), "label": DERAILED if source.get("has_removed_comment") else CIVIL}
        row.update(conversation_features(conversation, first_n=min(first_n, len(ordered))))
        rows.append(row)
    return pd.DataFrame(rows)

cutoff_rows = []
for cutoff in [2, 3, 4, 5, 6]:
    prefix = prefix_structural_examples(path, cutoff)
    prefix_train = prefix[prefix["split"] == "train"]
    prefix_val = prefix[prefix["split"] == "val"]
    cols = structural_columns(prefix_train, include_votes=False)
    model = feature_model(cols).fit(prefix_train, prefix_train["label"])
    result = score(model, prefix_val, prefix_val)[2]
    result["first_n"] = cutoff
    cutoff_rows.append(result)
pd.DataFrame(cutoff_rows).set_index("first_n").round(4)


,accuracy,macro_f1,roc_auc,paired_acc
first_n,,,,
2,0.5614,0.5579,0.5719,0.5687
3,0.5775,0.5772,0.6037,0.6140
4,0.6009,0.5982,0.6200,0.6491
5,0.5994,0.5974,0.6205,0.6491
6,0.6001,0.5991,0.6300,0.6608


## Conclusion

Earlier prefixes contain useful signal, but performance increases as more context becomes available. The best cutoff in this sweep is `first_n=6`, with accuracy 0.6001, macro F1 0.5991, ROC AUC 0.6300, and paired accuracy 0.6608. Every cutoff is below the full-context headline baseline, so this experiment does not improve the main model. It should be reported separately as an early-forecasting analysis, not as a failed replacement for the headline task.


## Optional candidate — discourse-act aggregates

Notebook 09 shows that parent context substantially improves discourse-act classification, while the clustering notebook found that topic is not a useful derailment predictor. A principled optional experiment is therefore to run the selected discourse classifier on CGA-CMV prefixes and aggregate behavior-oriented acts such as disagreement, negative reaction, agreement, appreciation, and elaboration.

This should be treated as a domain-shift experiment: the classifier was trained on Coarse Discourse, not CGA-CMV. Do not interpret its predicted acts as ground-truth labels. The code cell below is a schema placeholder for an exported per-comment prediction file and is intentionally disabled until the team agrees on the export format.


In [8]:
# Optional, not run in this experiment: discourse-act aggregates.
# Required export columns, if this experiment is pursued later, are:
# utterance_id, conversation_id, split, predicted_label, probability.
# The export must be generated without using CGA-CMV outcomes.

DISCOURSE_EXPORT = path.parents[1] / "discourse_predictions_on_cga_cmv.csv"

def aggregate_discourse_predictions(predictions, prefix_ids):
    selected = predictions[predictions["utterance_id"].isin(prefix_ids)].copy()
    if selected.empty:
        return pd.Series(dtype=float)
    counts = selected["predicted_label"].value_counts(normalize=True)
    row = {f"discourse_share_{label}": value for label, value in counts.items()}
    row["discourse_mean_confidence"] = selected["probability"].mean()
    row["discourse_n_comments"] = len(selected)
    return pd.Series(row)

# This experiment was not run, so it does not affect model selection.


## Conclusion

No discourse-act aggregate results were produced because this cell remains an optional template. It should not affect model selection. If attempted later, the predicted acts should be treated as noisy, domain-shifted behavioral indicators rather than ground-truth CGA-CMV labels.


## Validation-based candidate selection

The validation results have now been reviewed and recorded below. The selected configuration was frozen before the held-out comparison in the appended section.


In [ ]:
experiment_log = pd.DataFrame([
    {"experiment": "headline baseline", "status": "reference", "notes": "structural + reply-score summaries; test reference"},
    {"experiment": "richer reply summaries", "status": "rejected", "notes": "slightly worse overall than baseline"},
    {"experiment": "log structural features", "status": "selected candidate", "notes": "best validation result; evaluated on held-out test"},
    {"experiment": "word + character TF-IDF", "status": "rejected", "notes": "weaker than numeric baseline"},
    {"experiment": "numeric/text blend", "status": "not selected", "notes": "small validation gain; weaker than selected candidate"},
    {"experiment": "prefix cutoff", "status": "separate task", "notes": "early forecasting; not comparable to the full-context model"},
    {"experiment": "discourse-act aggregates", "status": "not run", "notes": "optional future experiment with domain shift"},
])
experiment_log


## Provisional selection conclusion

The log-transformed structural model with richer reply summaries remains the leading candidate. It is clearly stronger on validation than the headline baseline, while the text model, score blend, and cutoff sweep do not match it. The richer reply summaries alone slightly regress, so their value appears to come mainly from interacting with the transformed structural features.

The candidate has not yet been evaluated on the held-out test set. The test and bootstrap cells below currently evaluate only the headline baseline, so no final improvement claim should be made yet.


## Held-out test evaluation

The original baseline test result is retained here as a reference. The appended section below evaluates the validation-selected candidate against this same baseline on the held-out test set.


In [ ]:
SELECTED_NAME = "headline baseline reference"

headline_test = score(headline, test, test)[2]
pd.DataFrame([headline_test], index=[SELECTED_NAME]).round(4)


## Conclusion

The headline baseline scores 0.6111 accuracy and macro F1, 0.6556 ROC AUC, and 0.6623 paired accuracy on the held-out test set. Its pair-aware 95% bootstrap intervals are:

- Accuracy: 0.5863–0.6360
- Macro F1: 0.5862–0.6359
- ROC AUC: 0.6268–0.6838
- Paired accuracy: 0.6257–0.6988

These results provide the final comparison point. The provisional log-feature candidate still needs to be evaluated on this same test set before its improvement can be assessed.


## Pair-aware bootstrap confidence intervals

Because CGA-CMV is made of matched pairs, resample pairs rather than individual rows. The interval code below reports uncertainty for accuracy, macro F1, ROC AUC, and paired accuracy. Use the same function for the selected candidate and the headline baseline.


In [11]:
def pair_key(frame):
    source = frame["conversation_id"].astype(str).str.rsplit(":", n=1).str[-1]
    partner = frame["pair_id"].astype(str)
    return pd.Series(["|".join(sorted([a, b])) for a, b in zip(source, partner)], index=frame.index)

def pair_bootstrap(frame, scores, n_boot=2000, seed=42):
    frame = frame.reset_index(drop=True)
    scores = np.asarray(scores)
    keys = pair_key(frame).to_numpy()
    unique_pairs = np.unique(keys)
    rng = np.random.default_rng(seed)
    rows = []
    for _ in range(n_boot):
        sampled = rng.choice(unique_pairs, size=len(unique_pairs), replace=True)
        indices = np.concatenate([np.flatnonzero(keys == key) for key in sampled])
        sample = frame.iloc[indices]
        s = scores[indices]
        y = sample["label"].to_numpy()
        pred = np.where(s >= 0.5, CIVIL, DERAILED)
        rows.append({"accuracy": accuracy_score(y, pred), "macro_f1": f1_score(y, pred, average="macro"), "roc_auc": roc_auc_score(y == CIVIL, s), "paired_acc": paired_accuracy(s, sample["label"].tolist(), sample["pair_id"].tolist(), sample["conversation_id"].tolist())})
    estimates = pd.DataFrame(rows)
    return pd.DataFrame({"estimate": estimates.mean(), "lower_95": estimates.quantile(0.025), "upper_95": estimates.quantile(0.975)})

headline_test_score = civil_probability(headline, test)
pair_bootstrap(test, headline_test_score)


,estimate,lower_95,upper_95
accuracy,0.611301,0.586257,0.635965
macro_f1,0.611200,0.586243,0.635888
roc_auc,0.655851,0.626817,0.683777
paired_acc,0.662455,0.625731,0.698830


## Conclusion

The pair-aware bootstrap confirms substantial uncertainty around the baseline metrics on 1,368 matched-pair test conversations. In particular, the intervals are wide enough that modest validation differences should not be treated as conclusive without evaluating the selected candidate under the same procedure.


## Error analysis and recommendation

Use the next cell to inspect false positives, false negatives, and disagreements between the selected candidate and the headline baseline. The final written conclusion should state what helped, what did not help, whether the confidence interval supports a meaningful improvement, and whether the feature is appropriate for a “conversation health” signal.


In [12]:
test_review = test[["conversation_id", "pair_id", "label", "n_context", "context_text"]].copy()
test_review["headline_quality_score"] = headline_test_score
test_review["headline_prediction"] = np.where(test_review["headline_quality_score"] >= 0.5, CIVIL, DERAILED)

print("Civil conversations most likely to derail:")
display(test_review[test_review["label"] == CIVIL].nsmallest(10, "headline_quality_score"))
print("\nDerailed conversations most likely to remain civil:")
display(test_review[test_review["label"] == DERAILED].nlargest(10, "headline_quality_score"))


Civil conversations most likely to derail:


,conversation_id,pair_id,label,n_context,context_text,headline_quality_score,headline_prediction
4519,reddit:conversations-gone-awry-cmv-corpus:conv...,dsy3mza,civil,2,God(s) doesn't exist for the same reason flyin...,0.151111,derailed
2874,reddit:conversations-gone-awry-cmv-corpus:conv...,dg7x3eu,civil,10,>I haven't really seen any serious academic su...,0.156760,derailed
1833,reddit:conversations-gone-awry-cmv-corpus:conv...,d8n3ax4,civil,2,Would you agree that being hit with napalm and...,0.157912,derailed
1011,reddit:conversations-gone-awry-cmv-corpus:conv...,d2k2nkl,civil,7,How can your view change on this? It's built o...,0.165521,derailed
1976,reddit:conversations-gone-awry-cmv-corpus:conv...,d9pjo2p,civil,6,Some people are against Satanism because they ...,0.166760,derailed
1108,reddit:conversations-gone-awry-cmv-corpus:conv...,d3hd35h,civil,7,">I've read on many travel blogs, that Whites g...",0.175496,derailed
1376,reddit:conversations-gone-awry-cmv-corpus:conv...,d57vnxd,civil,4,"This is, of course, a hypothetical CMV which c...",0.206436,derailed
2244,reddit:conversations-gone-awry-cmv-corpus:conv...,dbns378,civil,9,If fractionalization is the worst multicultura...,0.209680,derailed
1817,reddit:conversations-gone-awry-cmv-corpus:conv...,d8l2h5w,civil,1,"You're not giving your organs to ""society"", yo...",0.220142,derailed
892,reddit:conversations-gone-awry-cmv-corpus:conv...,d1hnjaa,civil,8,I had to look up a video of it as all I saw wa...,0.221745,derailed



Derailed conversations most likely to remain civil:


,conversation_id,pair_id,label,n_context,context_text,headline_quality_score,headline_prediction
3243,reddit:conversations-gone-awry-cmv-corpus:conv...,djn1l9o,derailed,5,What the hell is wrong with you? Referring to ...,0.805166,civil
2071,reddit:conversations-gone-awry-cmv-corpus:conv...,dah4v0w,derailed,1,Jill Stein is motivated only by getting attent...,0.798555,civil
5800,reddit:conversations-gone-awry-cmv-corpus:conv...,e1945qd,derailed,1,Negative feedback is not the same as useful fe...,0.797908,civil
758,reddit:conversations-gone-awry-cmv-corpus:conv...,d0mknfb,derailed,2,My wife and I have been married for five years...,0.793895,civil
697,reddit:conversations-gone-awry-cmv-corpus:conv...,d0aa0j1,derailed,1,Cows were domesticated and bred solely for the...,0.792714,civil
5938,reddit:conversations-gone-awry-cmv-corpus:conv...,e294gjv,derailed,2,The real questions are - what is oppressive an...,0.781347,civil
1427,reddit:conversations-gone-awry-cmv-corpus:conv...,d5mzsxx,derailed,8,[deleted]\n\nSurely it needs the same as my me...,0.778500,civil
5844,reddit:conversations-gone-awry-cmv-corpus:conv...,e1nl9ov,derailed,1,This week a global report just came out in whi...,0.774046,civil
4422,reddit:conversations-gone-awry-cmv-corpus:conv...,ds91trg,derailed,4,By not restricting some people’s freedom to be...,0.767316,civil
5851,reddit:conversations-gone-awry-cmv-corpus:conv...,e1q1901,derailed,2,So the only thing it would take to convince yo...,0.760372,civil


## Current conclusion

The log-transformed structural model plus richer reply-score summaries was the strongest validation candidate and also improved every held-out test point estimate: accuracy and macro F1 rose from 0.6111 to 0.6177, ROC AUC from 0.6556 to 0.6690, and paired accuracy from 0.6623 to 0.6711.

The gains are modest, and the pair-aware bootstrap intervals overlap, so the result should not be presented as a definitive or statistically established improvement. It is reasonable to use this model as the preferred documented conversation-health signal, while clearly reporting the uncertainty and the fact that the model remains probabilistic rather than a reliable standalone classifier.



---

# Test evaluations

This section is kept separate because it uses the held-out test set only after the validation-based model selection is complete. Keeping this evaluation distinct helps prevent test results from influencing feature selection, model tuning, or the earlier development comparisons.

The section compares the selected candidate against the headline baseline and computes pair-aware bootstrap confidence intervals for both models.


In [13]:
# Rebuild both models from training data only.
# The test set is used only in the evaluation cells below.

test_rich = add_reply_summary_features(test)
test_log = add_log_structural_features(test_rich)

selected_model = feature_model(log_cols + rich_reply_cols).fit(
    train_log, train_log["label"]
)
selected_name = "log structural + rich reply summaries"

confirmed_baseline = feature_model(headline_cols).fit(
    train, train["label"]
)

selected_test_result, selected_test_quality, selected_test_metrics = score(
    selected_model, test_log, test
)
baseline_test_result, baseline_test_quality, baseline_test_metrics = score(
    confirmed_baseline, test, test
)

test_comparison = pd.DataFrame(
    [baseline_test_metrics, selected_test_metrics],
    index=["headline baseline", selected_name],
)
test_comparison.round(4)


,accuracy,macro_f1,roc_auc,paired_acc
headline baseline,0.6111,0.6111,0.6556,0.6623
log structural + rich reply summaries,0.6177,0.6177,0.6690,0.6711



## Test comparison interpretation

The selected candidate improves all four held-out point estimates: accuracy and macro F1 increase by 0.0066, ROC AUC by 0.0134, and paired accuracy by 0.0088. This is consistent with the validation result, but the gains are modest and the pair-aware 95% intervals overlap, so the test results support a cautious improvement claim rather than a conclusive one.


In [14]:
def appended_pair_key(frame):
    source = frame["conversation_id"].astype(str).str.rsplit(":", n=1).str[-1]
    partner = frame["pair_id"].astype(str)
    return pd.Series(
        ["|".join(sorted([a, b])) for a, b in zip(source, partner)],
        index=frame.index,
    )


def appended_pair_bootstrap(frame, scores, n_boot=2000, seed=42):
    """Bootstrap matched pairs rather than individual conversations."""
    frame = frame.reset_index(drop=True)
    scores = np.asarray(scores)
    keys = appended_pair_key(frame).to_numpy()
    unique_pairs = np.unique(keys)
    rng = np.random.default_rng(seed)
    rows = []

    for _ in range(n_boot):
        sampled_pairs = rng.choice(unique_pairs, size=len(unique_pairs), replace=True)
        indices = np.concatenate(
            [np.flatnonzero(keys == key) for key in sampled_pairs]
        )
        sample = frame.iloc[indices]
        sample_scores = scores[indices]
        labels = sample["label"].to_numpy()
        predictions = np.where(sample_scores >= 0.5, CIVIL, DERAILED)

        rows.append({
            "accuracy": accuracy_score(labels, predictions),
            "macro_f1": f1_score(labels, predictions, average="macro"),
            "roc_auc": roc_auc_score(labels == CIVIL, sample_scores),
            "paired_acc": paired_accuracy(
                sample_scores,
                sample["label"].tolist(),
                sample["pair_id"].tolist(),
                sample["conversation_id"].tolist(),
            ),
        })

    bootstrap_metrics = pd.DataFrame(rows)
    return pd.DataFrame({
        "estimate": bootstrap_metrics.mean(),
        "lower_95": bootstrap_metrics.quantile(0.025),
        "upper_95": bootstrap_metrics.quantile(0.975),
    })

print("Headline baseline — pair-aware 95% intervals")
display(appended_pair_bootstrap(test, baseline_test_quality))

print(f"{selected_name} — pair-aware 95% intervals")
display(appended_pair_bootstrap(test, selected_test_quality))


Headline baseline — pair-aware 95% intervals


,estimate,lower_95,upper_95
accuracy,0.611301,0.586257,0.635965
macro_f1,0.611200,0.586243,0.635888
roc_auc,0.655851,0.626817,0.683777
paired_acc,0.662455,0.625731,0.698830


log structural + rich reply summaries — pair-aware 95% intervals


,estimate,lower_95,upper_95
accuracy,0.617944,0.592836,0.643275
macro_f1,0.617871,0.592676,0.643238
roc_auc,0.669175,0.642075,0.696088
paired_acc,0.671575,0.635965,0.706140



## Final candidate decision

The selected candidate may be adopted as the preferred model for the report's probabilistic “conversation health” signal because it improves every tested metric on the held-out set. The report should describe the improvement as modest, include the bootstrap intervals, and avoid implying that the model can reliably determine whether an individual conversation will derail.


In [15]:
# Final decision recorded after reviewing the held-out test results.
final_decision = pd.DataFrame([{
    "selected_candidate": selected_name,
    "decision": "Adopt provisionally; it improves all held-out point estimates, but the gain is modest and intervals overlap.",
    "include_as_conversation_health_signal": "Yes, as a probabilistic and documented signal with uncertainty caveats.",
}])
final_decision


,selected_candidate,decision,include_as_conversation_health_signal
0,log structural + rich reply summaries,Adopt provisionally; it improves all held-out ...,"Yes, as a probabilistic and documented signal ..."


## Final remark

Overall, these experiments did not produce a substantial improvement: test accuracy, ROC AUC, and paired accuracy each increased by only about one percentage point or less, so the selected model should be viewed as a modest refinement rather than a significant advancement.
